# Diagnostic: Why the Heterodyned TaylorF2 Run Produced Inaccurate Results

This notebook investigates why `PhaseMarg_Heterodyned_TaylorF2.csv` (GWOSC data + TaylorF2 + 100 bins + fmax=2048)
produced wildly inaccurate chirp-mass and H0 results, while the two `kazewong` runs gave LVK-consistent results.

## Summary of runs
| Run | Data Source | Waveform | N_BINS | fmax (Hz) | Output |
|-----|------------|----------|--------|-----------|--------|
| Heterodyned | GWOSC | IMRPhenomD_NRTv2 | 100 | 2048 | `PhaseMarg_Heterodyned.csv` |
| Kazewong | kazewong/bilby | IMRPhenomD_NRTv2 | 501 | 1792 | `PhaseMarg_Heterodyned_Kazewong.csv` |
| **TaylorF2** | **GWOSC** | **TaylorF2** | **100** | **2048** | **`PhaseMarg_Heterodyned_TaylorF2.csv`** |
| Kazewong_TaylorF2 | kazewong/bilby | TaylorF2 | 501 | 1792 | `PhaseMarg_Heterodyned_Kazewong_TaylorF2.csv` |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import h5py
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.interpolate import interp1d

mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'cm'
mpl.rcParams['figure.dpi'] = 120

print('Setup complete')

## 1. Load All Posterior Results + LVK Reference

In [ ]:
from anesthetic import read_chains, MCMCSamples

# Load the 4 runs
runs = {
    'Heterodyned (IMRPhenomD, GWOSC, 100 bins)': 'Results/PhaseMarg_Heterodyned.csv',
    'Kazewong (IMRPhenomD, bilby, 501 bins)': 'Results/PhaseMarg_Heterodyned_Kazewong.csv',
    'TaylorF2 (GWOSC, 100 bins) [BAD]': 'Results/PhaseMarg_Heterodyned_TaylorF2.csv',
    'Kazewong+TaylorF2 (bilby, 501 bins)': 'Results/PhaseMarg_Heterodyned_Kazewong_TaylorF2.csv',
}

samples = {}
for label, path in runs.items():
    samples[label] = read_chains(path)
    n = len(samples[label])
    print(f'{label}: {n} samples')

# Load LVK (GWTC-1) posteriors
GWTC1_PATH = 'Results/GW170817_GWTC-1.hdf5'
with h5py.File(GWTC1_PATH, 'r') as f:
    lvk_data = f['IMRPhenomPv2NRT_lowSpin_posterior'][:]

m1_lvk = lvk_data['m1_detector_frame_Msun']
m2_lvk = lvk_data['m2_detector_frame_Msun']
Mc_lvk = (m1_lvk * m2_lvk)**0.6 / (m1_lvk + m2_lvk)**0.2
q_lvk = m2_lvk / m1_lvk
dL_lvk = lvk_data['luminosity_distance_Mpc']
iota_lvk = np.arccos(lvk_data['costheta_jn'])

print(f'LVK: {len(Mc_lvk)} samples')
print(f'  Mc: {np.median(Mc_lvk):.4f}, q: {np.median(q_lvk):.4f}, dL: {np.median(dL_lvk):.1f}')

## 2. Summary Statistics — Quantifying the Discrepancy

In [ ]:
params_to_check = ['M_c', 'q', 'd_L', 'iota', 'H_0']
print(f'{"Run":55s} | {"M_c":>10s} | {"q":>10s} | {"d_L":>10s} | {"iota":>10s} | {"H_0":>10s} | {"logL_max":>10s}')
print('-' * 130)

# LVK reference
print(f'{"LVK (GWTC-1)":55s} | {np.median(Mc_lvk):10.4f} | {np.median(q_lvk):10.4f} | '
      f'{np.median(dL_lvk):10.1f} | {np.median(iota_lvk):10.3f} | {"N/A":>10s} | {"N/A":>10s}')

for label, s in samples.items():
    vals = {p: np.median(s[p].to_numpy()) for p in params_to_check}
    logL_max = s['logL'].to_numpy().max()
    marker = ' <<<' if 'BAD' in label else ''
    print(f'{label:55s} | {vals["M_c"]:10.4f} | {vals["q"]:10.4f} | '
          f'{vals["d_L"]:10.1f} | {vals["iota"]:10.3f} | {vals["H_0"]:10.1f} | {logL_max:10.1f}{marker}')

## 3. Posterior Overlays: Chirp Mass

In [ ]:
colors = {'Heterodyned (IMRPhenomD, GWOSC, 100 bins)': 'C0',
          'Kazewong (IMRPhenomD, bilby, 501 bins)': 'C2',
          'TaylorF2 (GWOSC, 100 bins) [BAD]': 'red',
          'Kazewong+TaylorF2 (bilby, 501 bins)': 'C4'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for param, ax, xlbl in zip(['M_c', 'd_L', 'q'],
                            axes,
                            [r'$\mathcal{M}_c$ ($M_\odot$)', r'$d_L$ (Mpc)', r'$q$']):
    # LVK
    if param == 'M_c':
        lvk_vals = Mc_lvk
    elif param == 'd_L':
        lvk_vals = dL_lvk
    elif param == 'q':
        lvk_vals = q_lvk
    
    kde_lvk = gaussian_kde(lvk_vals)
    x_range = np.linspace(np.percentile(lvk_vals, 1), np.percentile(lvk_vals, 99), 500)
    ax.fill_between(x_range, kde_lvk(x_range), alpha=0.2, color='grey', label='LVK (GWTC-1)')
    ax.plot(x_range, kde_lvk(x_range), 'k--', alpha=0.5, lw=1.5)
    
    for label, s in samples.items():
        vals = s[param].to_numpy()
        weights = np.asarray(s.get_weights())
        weights = weights / weights.sum()
        mask = np.isfinite(vals) & np.isfinite(weights) & (weights > 0)
        vals, weights = vals[mask], weights[mask]
        
        try:
            kde = gaussian_kde(vals, weights=weights)
            lw = 3 if 'BAD' in label else 1.5
            ls = '-' if 'BAD' not in label else '-'
            ax.plot(x_range, kde(x_range), color=colors[label], lw=lw, label=label.split('(')[0].strip())
        except:
            pass
    
    ax.set_xlabel(xlbl, fontsize=13)
    ax.set_ylabel('Density', fontsize=13)

axes[0].legend(fontsize=8, loc='upper right')
fig.suptitle('Posterior Comparison: Key Parameters', fontsize=14, y=1.02)
fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_posteriors_Mc_dL_q.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. H0 Posterior Overlay (with Planck & SHoES)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x_h0 = np.linspace(20, 140, 1000)

for label, s in samples.items():
    h0 = s['H_0'].to_numpy()
    weights = np.asarray(s.get_weights())
    mask = np.isfinite(h0) & np.isfinite(weights) & (weights > 0) & (h0 >= 20) & (h0 <= 140)
    h0, weights = h0[mask], weights[mask]
    weights /= weights.sum()
    
    kde = gaussian_kde(h0, weights=weights)
    pdf = kde(x_h0)
    map_val = x_h0[np.argmax(pdf)]
    
    lw = 3 if 'BAD' in label else 2
    ax.plot(x_h0, pdf, color=colors[label], lw=lw, label=f'{label.split("(")[0].strip()} (MAP={map_val:.0f})')
    ax.fill_between(x_h0, pdf, alpha=0.1, color=colors[label])

# Reference bands
ax.axvspan(66.93-0.62, 66.93+0.62, alpha=0.3, color='#0CDE79', edgecolor='none', label='Planck')
ax.axvspan(73.24-1.74, 73.24+1.74, alpha=0.3, color='#E87317', edgecolor='none', label='SHoES')

ax.set_xlabel(r'$H_0$ (km s$^{-1}$ Mpc$^{-1}$)', fontsize=14)
ax.set_ylabel(r'$P(H_0)$', fontsize=14)
ax.set_xlim(20, 140)
ax.legend(fontsize=9)
ax.set_title('H0 Posterior: TaylorF2+GWOSC run is biased low', fontsize=13)
fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_H0_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Check 1: Data Integrity — Kazewong vs GWOSC

Both datasets use the same sampling rate (4096 Hz), duration (128 s), and frequency resolution (df=0.0078125 Hz).
The kazewong data comes from bilby's preprocessing pipeline.

In [ ]:
# Load GWOSC HDF5 strain data (4096 Hz, CLN version)
gps = 1187008882.43
gwosc_files = {
    'H1': 'EventData/GWOSC/GW170817/H-H1_LOSC_CLN_4_V1-1187007040-2048.hdf5',
    'L1': 'EventData/GWOSC/GW170817/L-L1_LOSC_CLN_4_V1-1187007040-2048.hdf5',
    'V1': 'EventData/GWOSC/GW170817/V-V1_LOSC_CLN_4_V1-1187007040-2048.hdf5',
}

gwosc_strain = {}
for det, fpath in gwosc_files.items():
    with h5py.File(fpath, 'r') as f:
        strain_data = f['strain/Strain'][:]
        gps_start = f['meta/GPSstart'][()]
        duration = f['meta/Duration'][()]
    fs = len(strain_data) / duration  # sampling rate
    t = np.arange(len(strain_data)) / fs + gps_start
    gwosc_strain[det] = {'t': t, 'strain': strain_data, 'fs': fs, 'gps_start': gps_start}
    print(f'{det}: {len(strain_data)} samples, fs={fs:.0f} Hz, GPS=[{gps_start}, {gps_start+duration}]')

# --- Time-domain plots around merger ---
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
window = 1.0  # seconds around merger

for ax, det in zip(axes, ['H1', 'L1', 'V1']):
    d = gwosc_strain[det]
    mask = (d['t'] >= gps - window) & (d['t'] <= gps + 0.1)
    ax.plot(d['t'][mask] - gps, d['strain'][mask] * 1e21, lw=0.5, alpha=0.8)
    ax.set_ylabel(f'{det} strain (x1e-21)', fontsize=11)
    ax.axvline(0, color='red', ls='--', alpha=0.5, label='GPS trigger')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Time relative to trigger (s)', fontsize=12)
axes[0].set_title('GWOSC CLN 4096 Hz strain around GW170817 merger', fontsize=13)
fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_GWOSC_strain_TD.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Spectrograms around merger ---
from scipy.signal import spectrogram

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, det in zip(axes, ['H1', 'L1', 'V1']):
    d = gwosc_strain[det]
    # Extract ~30s around merger
    seg_start = gps - 15
    seg_end = gps + 2
    mask = (d['t'] >= seg_start) & (d['t'] <= seg_end)
    seg_strain = d['strain'][mask]
    seg_t0 = d['t'][mask][0]
    
    f_spec, t_spec, Sxx = spectrogram(seg_strain, fs=d['fs'], nperseg=512, noverlap=480)
    
    # Limit frequency range
    f_mask = f_spec <= 500
    im = ax.pcolormesh(t_spec + (seg_t0 - gps), f_spec[f_mask], 
                       10*np.log10(Sxx[f_mask] + 1e-50), 
                       shading='gouraud', cmap='viridis', vmin=-250, vmax=-200)
    ax.axvline(0, color='red', ls='--', lw=1, alpha=0.8)
    ax.set_ylabel(f'{det}\nFreq (Hz)', fontsize=10)
    ax.set_ylim(20, 500)
    plt.colorbar(im, ax=ax, label='PSD (dB)')

axes[-1].set_xlabel('Time relative to trigger (s)', fontsize=12)
axes[0].set_title('Spectrogram: GWOSC CLN 4096 Hz around GW170817', fontsize=13)
fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_GWOSC_spectrogram.png', dpi=150, bbox_inches='tight')
plt.show()

## 5b. GWOSC Time-Domain Strain and Spectrograms

Load the raw GWOSC CLN (cleaned) strain data at 4096 Hz and produce:
- Time-domain plots around merger
- Spectrograms around merger

In [ ]:
import os

# Load kazewong data for H1
kz_prefix = 'EventData/kazewong/GW170817-IMRD_data0_1187008882-43_generation_data_dump.pickle'
detectors_names = ['H1', 'L1', 'V1']

kz_strain = {}
kz_psd = {}
for det in detectors_names:
    s = np.loadtxt(f'{kz_prefix}_{det}_fd_strain.txt')
    p = np.loadtxt(f'{kz_prefix}_{det}_psd.txt')
    kz_strain[det] = {'freq': s[:, 0], 're': s[:, 1], 'im': s[:, 2]}
    kz_psd[det] = {'freq': p[:, 0], 'psd': p[:, 1]}

print('Kazewong data properties:')
for det in detectors_names:
    f = kz_strain[det]['freq']
    print(f'  {det}: freq=[{f[0]:.3f}, {f[-1]:.3f}] Hz, df={f[1]-f[0]:.6f} Hz, '
          f'n_freq={len(f)}, duration={1/(f[1]-f[0]):.0f}s')
    psd = kz_psd[det]['psd']
    valid = np.isfinite(psd) & (psd > 0)
    print(f'       PSD finite bins: {valid.sum()}/{len(psd)}, '
          f'range (valid): [{psd[valid].min():.2e}, {psd[valid].max():.2e}]')

## 6. Check 2: PSD Comparison — Kazewong (bilby) vs Expected GWOSC Welch

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, det in zip(axes, detectors_names):
    freq = kz_psd[det]['freq']
    psd = kz_psd[det]['psd']
    
    # Only plot valid range
    mask = (freq >= 10) & (freq <= 2048) & np.isfinite(psd) & (psd > 0)
    ax.loglog(freq[mask], np.sqrt(psd[mask]), label=f'{det} kazewong (bilby)', alpha=0.8)
    
    ax.set_xlabel('Frequency (Hz)', fontsize=12)
    ax.set_ylabel(r'ASD ($\mathrm{Hz}^{-1/2}$)', fontsize=12)
    ax.set_title(det, fontsize=13)
    ax.axvline(23, color='grey', ls='--', alpha=0.5, label='fmin=23 Hz')
    ax.axvline(1792, color='orange', ls='--', alpha=0.5, label='fmax=1792 (kazewong)')
    ax.axvline(2048, color='red', ls='--', alpha=0.5, label='fmax=2048 (GWOSC)')
    ax.legend(fontsize=8)
    ax.set_xlim(10, 3000)

fig.suptitle('PSD (ASD) from kazewong/bilby data', fontsize=14, y=1.02)
fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_PSD.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.signal import welch as scipy_welch

# Compute GWOSC Welch PSD from off-source segment (matching script approach)
# Script config: psd_duration=1024, psd_pad=16, analysis start = gps - 126
duration_analysis = 128
post_trigger = 2
psd_duration = 1024
psd_pad = 16

analysis_start = gps - (duration_analysis - post_trigger)
psd_start = analysis_start - psd_pad - psd_duration
psd_end = analysis_start - psd_pad

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, det in zip(axes, ['H1', 'L1', 'V1']):
    d = gwosc_strain[det]
    
    # Extract PSD segment
    psd_mask = (d['t'] >= psd_start) & (d['t'] < psd_end)
    psd_segment = d['strain'][psd_mask]
    
    nperseg_welch = int(duration_analysis * d['fs'])  # matches script: psd_fftlength
    f_welch, psd_welch = scipy_welch(psd_segment, fs=d['fs'], nperseg=nperseg_welch)
    
    # Plot GWOSC Welch
    valid_w = (f_welch >= 10) & (f_welch <= 2048) & (psd_welch > 0)
    ax.loglog(f_welch[valid_w], np.sqrt(psd_welch[valid_w]), 'C0', alpha=0.7, lw=1.2, label=f'GWOSC Welch')
    
    # Plot kazewong
    freq_kz = kz_psd[det]['freq']
    psd_kz = kz_psd[det]['psd']
    valid_k = (freq_kz >= 10) & (freq_kz <= 2048) & np.isfinite(psd_kz) & (psd_kz > 0)
    ax.loglog(freq_kz[valid_k], np.sqrt(psd_kz[valid_k]), 'C2', alpha=0.7, lw=1.2, label=f'kazewong (bilby)')
    
    ax.set_xlabel('Frequency (Hz)', fontsize=12)
    ax.set_ylabel(r'ASD (Hz$^{-1/2}$)', fontsize=12)
    ax.set_title(det, fontsize=13)
    ax.legend(fontsize=9)
    ax.set_xlim(10, 3000)
    ax.grid(alpha=0.2)

fig.suptitle('PSD Comparison: GWOSC Welch vs kazewong/bilby', fontsize=14, y=1.02)
fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_PSD_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6b. PSD Comparison: GWOSC Welch vs kazewong (bilby)

Compute Welch PSD from the raw GWOSC strain (matching the script's method) and overlay with kazewong PSD.

## 7. Check 3: Waveform Comparison — TaylorF2 vs IMRPhenomD_NRTidalv2

**Critical finding:** The TaylorF2 implementation has **no ISCO frequency cutoff**. It generates signal at all frequencies,
unlike IMRPhenomD which naturally tapers after merger+ringdown. This means:
- TaylorF2 extends to fmax=2048 Hz with increasingly unreliable PN phasing
- The valid frequency range for heterodyne bins is much wider for TaylorF2
- 100 bins must cover a wider range, making each bin coarser

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp

import sys
sys.path.insert(0, 'libraries/jim/src')
sys.path.insert(0, 'libraries/ripple/src')

from jimgw.core.single_event.waveform import RippleIMRPhenomD_NRTidalv2, RippleTaylorF2

# Reference parameters (GWTC-1 medians)
with h5py.File(GWTC1_PATH, 'r') as f:
    data = f['IMRPhenomPv2NRT_lowSpin_posterior'][:]
m1 = np.median(data['m1_detector_frame_Msun'])
m2 = np.median(data['m2_detector_frame_Msun'])
M = m1 + m2
eta_val = m1 * m2 / M**2
Mc_ref = M * eta_val**(3/5)
q_ref = m2 / m1

gps = 1187008882.43
from astropy.time import Time
gmst = Time(gps, format='gps').sidereal_time('apparent', 'greenwich').rad

ref_params = {
    'M_c': float(Mc_ref), 'q': float(q_ref), 'eta': float(min(eta_val, 0.249995)),
    's1_z': float(np.median(data['spin1'] * data['costilt1'])),
    's2_z': float(np.median(data['spin2'] * data['costilt2'])),
    'd_L': float(np.median(data['luminosity_distance_Mpc'])),
    'iota': float(np.median(np.arccos(data['costheta_jn']))),
    'ra': float(np.median(data['right_ascension'])),
    'dec': float(np.median(data['declination'])),
    'lambda_1': float(np.median(data['lambda1'])),
    'lambda_2': float(np.median(data['lambda2'])),
    't_c': 0.0, 'phase_c': 0.0, 'psi': 0.0,
    'trigger_time': float(gps), 'gmst': float(gmst),
}

print(f'Reference: Mc={ref_params["M_c"]:.4f}, q={ref_params["q"]:.4f}, dL={ref_params["d_L"]:.1f}')

# Generate waveforms
freqs = jnp.linspace(23.0, 2048.0, 10000)

wf_imr = RippleIMRPhenomD_NRTidalv2(f_ref=20.0, use_lambda_tildes=False, no_taper=False)
wf_tf2 = RippleTaylorF2(f_ref=20.0, use_lambda_tildes=False)

h_imr = wf_imr(freqs, ref_params)
h_tf2 = wf_tf2(freqs, ref_params)

# Compute amplitudes
amp_imr = np.sqrt(np.array(jnp.abs(h_imr['plus']))**2 + np.array(jnp.abs(h_imr['cross']))**2)
amp_tf2 = np.sqrt(np.array(jnp.abs(h_tf2['plus']))**2 + np.array(jnp.abs(h_tf2['cross']))**2)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Amplitude
axes[0].semilogy(np.array(freqs), amp_imr, label='IMRPhenomD_NRTidalv2', alpha=0.8)
axes[0].semilogy(np.array(freqs), amp_tf2, label='TaylorF2', alpha=0.8)
axes[0].axvline(1792, color='orange', ls='--', alpha=0.5, label='fmax=1792 (kazewong)')
axes[0].axvline(2048, color='red', ls='--', alpha=0.5, label='fmax=2048 (GWOSC)')
axes[0].set_ylabel('Waveform Amplitude', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_title('Waveform comparison at GWTC-1 median parameters', fontsize=13)

# Phase
phase_imr = np.unwrap(np.angle(np.array(h_imr['plus'])))
phase_tf2 = np.unwrap(np.angle(np.array(h_tf2['plus'])))

axes[1].plot(np.array(freqs), phase_imr, label='IMRPhenomD_NRTidalv2', alpha=0.8)
axes[1].plot(np.array(freqs), phase_tf2, label='TaylorF2', alpha=0.8)
axes[1].axvline(1792, color='orange', ls='--', alpha=0.5)
axes[1].axvline(2048, color='red', ls='--', alpha=0.5)
axes[1].set_xlabel('Frequency (Hz)', fontsize=12)
axes[1].set_ylabel('Phase (rad)', fontsize=12)
axes[1].legend(fontsize=10)

fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_waveform_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Where does IMRPhenomD effectively terminate?
valid_imr = amp_imr > 1e-30
if valid_imr.any():
    f_max_imr = float(np.array(freqs)[valid_imr][-1])
    print(f'IMRPhenomD_NRTv2 effective fmax (amp > 1e-30): {f_max_imr:.0f} Hz')
valid_tf2 = amp_tf2 > 1e-30
if valid_tf2.any():
    f_max_tf2 = float(np.array(freqs)[valid_tf2][-1])
    print(f'TaylorF2 effective fmax (amp > 1e-30): {f_max_tf2:.0f} Hz')

print(f'\nKey insight: TaylorF2 has signal at ALL frequencies (no ISCO cutoff)!')
print(f'IMRPhenomD naturally terminates after ringdown.')
print(f'With 100 bins and fmax=2048, TaylorF2 bins are ~5x coarser than with 501 bins and fmax=1792.')

## 8. Check 4: Heterodyne Bin Resolution Analysis

The relative binning (heterodyne) approximation assumes `h(f)/h_ref(f)` varies slowly within each bin.
With too few bins, this linear approximation breaks down — distorting the likelihood surface.

In [ ]:
def max_phase_diff(f, f_low, f_high, chi=1.0):
    """Maximum accumulated phase across PN orders."""
    gamma = np.arange(-5, 6, 1) / 3.0
    f_2d = np.repeat(f[:, None], len(gamma), axis=1)
    f_star = np.repeat(f_low, len(gamma))
    f_star[gamma >= 0] = f_high
    return 2 * np.pi * chi * np.sum((f_2d / f_star) ** gamma * np.sign(gamma), axis=1)

def make_binning_scheme(freqs, n_bins, chi=1):
    phase_diff_array = max_phase_diff(freqs, freqs[0], freqs[-1], chi=chi)
    bin_f = interp1d(phase_diff_array, freqs)
    f_bins = np.array([bin_f(i) for i in np.linspace(
        phase_diff_array[0], phase_diff_array[-1], n_bins + 1)])
    f_bins_center = (f_bins[:-1] + f_bins[1:]) / 2
    return f_bins, f_bins_center

# Compare binning for 100 vs 501 bins
freqs_analysis = np.linspace(23.0, 2048.0, 260000)  # GWOSC resolution
freqs_kz = np.linspace(23.0, 1792.0, 226000)  # kazewong resolution

bins_100, centers_100 = make_binning_scheme(freqs_analysis, 100)
bins_501_kz, centers_501_kz = make_binning_scheme(freqs_kz, 501)

# Bin widths
widths_100 = np.diff(bins_100)
widths_501 = np.diff(bins_501_kz)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Bin widths
axes[0].plot(centers_100, widths_100, 'ro-', ms=2, label=f'100 bins, 23-2048 Hz (GWOSC TaylorF2)', alpha=0.7)
axes[0].plot(centers_501_kz, widths_501, 'g.-', ms=1, label=f'501 bins, 23-1792 Hz (kazewong)', alpha=0.7)
axes[0].set_xlabel('Bin center frequency (Hz)', fontsize=12)
axes[0].set_ylabel('Bin width (Hz)', fontsize=12)
axes[0].set_title('Heterodyne bin widths: 100 bins (GWOSC) vs 501 bins (kazewong)', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_yscale('log')

# Phase accumulation per bin
phase_100 = max_phase_diff(freqs_analysis, freqs_analysis[0], freqs_analysis[-1])
total_phase = phase_100[-1] - phase_100[0]
phase_per_bin_100 = total_phase / 100

phase_kz = max_phase_diff(freqs_kz, freqs_kz[0], freqs_kz[-1])
total_phase_kz = phase_kz[-1] - phase_kz[0]
phase_per_bin_501 = total_phase_kz / 501

axes[1].bar(['100 bins\n(GWOSC, 23-2048 Hz)', '501 bins\n(kazewong, 23-1792 Hz)'],
           [phase_per_bin_100, phase_per_bin_501],
           color=['red', 'green'], alpha=0.7)
axes[1].set_ylabel('Phase per bin (rad)', fontsize=12)
axes[1].set_title(f'Average phase accumulation per bin\n'
                  f'100 bins: {phase_per_bin_100:.1f} rad/bin vs 501 bins: {phase_per_bin_501:.1f} rad/bin '
                  f'({phase_per_bin_100/phase_per_bin_501:.1f}x coarser)', fontsize=12)

fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_bin_resolution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total phase (100 bins, 23-2048 Hz): {total_phase:.0f} rad')
print(f'Total phase (501 bins, 23-1792 Hz): {total_phase_kz:.0f} rad')
print(f'Phase per bin (100): {phase_per_bin_100:.1f} rad/bin')
print(f'Phase per bin (501): {phase_per_bin_501:.1f} rad/bin')
print(f'Ratio: {phase_per_bin_100/phase_per_bin_501:.1f}x — bins are much coarser for the GWOSC+100 run')

## 9. Check 5: Heterodyne Ratio Smoothness Test

The relative binning approximation assumes h(θ)/h(θ_ref) is approximately linear within each bin.
Let's check how well this holds for TaylorF2 with 100 vs 501 bins at a perturbed parameter point.

In [ ]:
from jimgw.core.single_event.detector import get_H1

# Perturbed parameters (shift Mc by ~0.01 Msun, dL by +10 Mpc)
perturbed = dict(ref_params)
perturbed['M_c'] = ref_params['M_c'] + 0.01
perturbed['d_L'] = ref_params['d_L'] + 10.0
perturbed['eta'] = perturbed['q'] / (1 + perturbed['q'])**2

# Compute waveform ratio for TaylorF2
freqs_fine = jnp.linspace(23.0, 2048.0, 50000)

h_ref = wf_tf2(freqs_fine, ref_params)
h_pert = wf_tf2(freqs_fine, perturbed)

# Focus on plus polarization
ratio = np.array(h_pert['plus'] / h_ref['plus'])
freqs_np = np.array(freqs_fine)

# Mark bin edges for 100-bin scheme
bins_100_test, _ = make_binning_scheme(freqs_np, 100)
bins_501_test, _ = make_binning_scheme(freqs_np, 501)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Full range - amplitude of ratio
ax = axes[0, 0]
ax.plot(freqs_np, np.abs(ratio), 'b-', alpha=0.5, lw=0.5)
for b in bins_100_test:
    ax.axvline(b, color='red', alpha=0.15, lw=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('|h(perturbed)/h(ref)|')
ax.set_title('TaylorF2 ratio amplitude — 100-bin edges (red lines)')

# Zoom: high frequency
ax = axes[0, 1]
hf_mask = (freqs_np > 800) & (freqs_np < 1500)
ax.plot(freqs_np[hf_mask], np.abs(ratio[hf_mask]), 'b-', alpha=0.7, lw=0.8)
for b in bins_100_test:
    if 800 < b < 1500:
        ax.axvline(b, color='red', alpha=0.3, lw=1, label='100-bin edge' if b == bins_100_test[(bins_100_test > 800) & (bins_100_test < 1500)][0] else '')
for b in bins_501_test:
    if 800 < b < 1500:
        ax.axvline(b, color='green', alpha=0.15, lw=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('|h(perturbed)/h(ref)|')
ax.set_title('Zoom: 800-1500 Hz — 100-bin (red) vs 501-bin (green) edges')

# Phase of ratio
ax = axes[1, 0]
ratio_phase = np.unwrap(np.angle(ratio))
ax.plot(freqs_np, ratio_phase, 'b-', alpha=0.5, lw=0.5)
for b in bins_100_test:
    ax.axvline(b, color='red', alpha=0.15, lw=0.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Phase of h(perturbed)/h(ref) (rad)')
ax.set_title('TaylorF2 ratio phase — 100-bin edges')

# Linearity check within a bin: pick a wide high-freq bin
ax = axes[1, 1]
# Find a bin at high frequency for 100-bin scheme
bin_idx = 80  # somewhere at high freq
if bin_idx < len(bins_100_test) - 1:
    f_lo, f_hi = bins_100_test[bin_idx], bins_100_test[bin_idx + 1]
    in_bin = (freqs_np >= f_lo) & (freqs_np <= f_hi)
    f_bin = freqs_np[in_bin]
    r_bin = ratio_phase[in_bin]
    # Linear fit
    if len(f_bin) > 2:
        coeffs = np.polyfit(f_bin, r_bin, 1)
        r_linear = np.polyval(coeffs, f_bin)
        residual = r_bin - r_linear
        ax.plot(f_bin, residual, 'b-', lw=1)
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('Residual from linear fit (rad)')
        ax.set_title(f'Non-linearity in bin {bin_idx} [{f_lo:.0f}-{f_hi:.0f} Hz]\n'
                     f'Max residual: {np.max(np.abs(residual)):.3f} rad')
        ax.axhline(0, color='k', ls='--', alpha=0.3)

fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_heterodyne_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Check 6: Log-Likelihood at Reference Point

The max logL difference between runs is very telling:
- Heterodyned (GWOSC, IMRPhenomD, 100 bins): logL_max ≈ 617
- Kazewong (IMRPhenomD, 501 bins): logL_max ≈ 534
- **TaylorF2 (GWOSC, 100 bins): logL_max ≈ 391** ← much lower!
- Kazewong+TaylorF2 (501 bins): logL_max ≈ 535

The TaylorF2+GWOSC run has a distorted likelihood surface, pushing the posterior away from the true region.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, s in samples.items():
    logL = s['logL'].to_numpy()
    logL = logL[np.isfinite(logL)]
    ax.hist(logL, bins=100, alpha=0.4, color=colors[label], 
            label=f'{label.split("(")[0].strip()} (max={logL.max():.0f})', density=True)

ax.set_xlabel('log-Likelihood', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Log-likelihood distribution across runs\n'
             'TaylorF2+GWOSC has a distorted (lower) likelihood surface', fontsize=12)
ax.legend(fontsize=8)
fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_logL_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Check 7: d_L vs H_0 Correlation — Identifying the Bias Mechanism

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, s in samples.items():
    dl = s['d_L'].to_numpy()
    h0 = s['H_0'].to_numpy()
    weights = np.asarray(s.get_weights())
    mask = np.isfinite(dl) & np.isfinite(h0) & np.isfinite(weights) & (weights > 0)
    dl, h0, w = dl[mask], h0[mask], weights[mask]
    
    # Subsample for visualization
    n_show = min(5000, len(dl))
    idx = np.random.choice(len(dl), n_show, p=w/w.sum(), replace=False)
    
    ms = 3 if 'BAD' in label else 1
    alpha = 0.5 if 'BAD' in label else 0.15
    axes[0].scatter(dl[idx], h0[idx], s=ms, alpha=alpha, color=colors[label], 
                    label=label.split('(')[0].strip())

axes[0].set_xlabel(r'$d_L$ (Mpc)', fontsize=12)
axes[0].set_ylabel(r'$H_0$ (km/s/Mpc)', fontsize=12)
axes[0].set_title('d_L vs H_0: TaylorF2+GWOSC biased to high d_L, low H_0', fontsize=11)
axes[0].legend(fontsize=8, markerscale=5)
axes[0].set_xlim(0, 200)
axes[0].set_ylim(20, 140)

# d_L marginal
x_dl = np.linspace(1, 200, 500)
for label, s in samples.items():
    dl = s['d_L'].to_numpy()
    weights = np.asarray(s.get_weights())
    mask = np.isfinite(dl) & np.isfinite(weights) & (weights > 0) & (dl > 0) & (dl < 200)
    dl, weights = dl[mask], weights[mask]
    weights /= weights.sum()
    kde = gaussian_kde(dl, weights=weights)
    lw = 3 if 'BAD' in label else 1.5
    axes[1].plot(x_dl, kde(x_dl), color=colors[label], lw=lw, label=label.split('(')[0].strip())

# LVK
kde_lvk_dl = gaussian_kde(dL_lvk)
axes[1].fill_between(x_dl, kde_lvk_dl(x_dl), alpha=0.2, color='grey', label='LVK')

axes[1].set_xlabel(r'$d_L$ (Mpc)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Distance posterior: TaylorF2+GWOSC biased high', fontsize=11)
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.savefig('Plots/Results/diagnostic_dL_H0.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Findings Summary

### Root Cause: Insufficient heterodyne bins for TaylorF2 with GWOSC data

The `PhaseMarg_Heterodyned_TaylorF2` run used **100 heterodyne bins** covering **23-2048 Hz**, while the successful kazewong runs used **501 bins** covering **23-1792 Hz**.

#### Why this matters specifically for TaylorF2:

1. **No ISCO cutoff**: The TaylorF2 implementation (`libraries/ripple/src/ripplegw/waveforms/TaylorF2.py`) generates non-zero signal at ALL frequencies — unlike IMRPhenomD which naturally tapers after merger+ringdown. This means the heterodyne setup's valid-frequency mask (`h_amp > 0`) doesn't trim the band, and 100 bins must span the entire 23-2048 Hz range.

2. **Phase accumulation**: With 100 bins over 2025 Hz, each bin accumulates ~5x more phase than with 501 bins over 1769 Hz. The linear interpolation `h/h_ref ≈ r0 + r1*(f - f_center)` used in relative binning breaks down when the ratio varies nonlinearly within a bin.

3. **Distorted likelihood surface**: The max log-likelihood for the TaylorF2+GWOSC run (391) is ~140 lower than the kazewong runs (~535), indicating the heterodyne approximation is systematically biasing the likelihood evaluation. This pushes the posterior toward higher d_L (median 68 vs 49 Mpc) and lower H_0 (median 46 vs 62 km/s/Mpc).

4. **Control experiment**: TaylorF2 with kazewong data (501 bins, fmax=1792) gives results consistent with the other runs, confirming the waveform itself is not the problem.

### Secondary factor: Data conditioning differences

While the GWOSC pipeline (simple Tukey window + Welch PSD) differs from bilby's more sophisticated preprocessing, this alone is not the cause — IMRPhenomD + GWOSC + 100 bins produces reasonable results. The data conditioning differences may contribute a smaller systematic offset.

### Recommended fixes (in priority order):

1. **Increase N_BINS to ≥500** for TaylorF2 runs in `GW170817_heterodyned_1.py` (simplest fix)
2. **Implement ISCO frequency cutoff** in `TaylorF2.py` to zero the waveform above f_ISCO ≈ 1/(6^{3/2} π M_total) — physically correct and reduces the effective bandwidth
3. **Match fmax to 1792 Hz** when using TaylorF2 to limit the frequency range to where PN phasing is reliable
4. **Use kazewong/bilby data** for all runs to eliminate data conditioning differences

## 13. Repro Commands

```bash
# Re-run TaylorF2 with more bins (fix applied):
# Edit GW170817_heterodyned_1.py: change N_BINS = 100 to N_BINS = 501
# Then:
python GW170817/Scripts/GW170817_heterodyned_1.py --waveform TaylorF2

# Or use kazewong data (already works):
python GW170817/Scripts/GW170817_heterodyned_kazewong.py --waveform TaylorF2

# Generate comparison plots:
python Plots/newplotter.py
python Plots/plot_H0.py

# Run this diagnostic notebook:
jupyter notebook diagnostics_GW170817.ipynb
```